In [15]:

from dotenv import load_dotenv
import os
from langchain_core.tools import tool
import json
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
import langchain_google_genai
from random import randint
import mysql.connector
from connections import get_product,get_connection


In [2]:
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

In [3]:
SYSTEM_PROMPT="""
Jesteś drugą hurtownią z ID H2 w łańcuchu dostaw. Twoim zadaniem jest przyjmowanie zamówień od producenta i przekazywanie ich do dwóch restauracji R1 i R2.
Na stanie masz 11 produktów flour, passata, mozzarella, parmigiano reggiano, burrata, buffala, prosciutto cotto, prosciutto crudo, arugula, lamb's lettuce, salami. 
Przy sprzedazy produktów, należy pamiętać o maksymalnych ilościach dostępnych na stanie, nie jesteś w stanie sprzedać więcej niż jest dostępne. Ponadto sprzedawać możesz tylko produkty, które są dostępne w hurtowni, w liczbach naturalnych . Nie możesz sprzedawać produktów, których nie masz na stanie.
Zawsze korzystaj z dostępnych narzędzi, aby badać stan faktyczny hurtowni i podejmować decyzje. Odpowiadaj rzeczowo i precyzyjnie w języku polskim
"""

In [ ]:
from fastmcp import FastMCP
from connections import get_connection, get_product

mcp = FastMCP("Warehouse H2")

@mcp.tool
def request_offer(product: str, quantity: int) -> dict:
    """
    Requests an offer for a specific product and quantity from the producer.
    """

    if quantity <= 0:
        return {
            "message": "Quantity must be greater than zero."
        }

    return {
        "sender_id": "H2",
        "receiver_id": "P",
        "message_type": "CALL_FOR_PROPOSAL",
        "item": {
            "name": product,
            "quantity": quantity
        }
    }


@mcp.tool
def accept_p_offer(product: str, quantity: int, price: float) -> dict:
    """
    Accepts an offer from the producer.
    """
    if quantity <= 0:
        return {
            "message": "Quantity must be greater than zero."
        }

    if price < 0:
        return {
            "message": "Price cannot be negative."
        }

    total_cost = quantity * price

    return {
        "sender_id": "H2",
        "receiver_id": "P",
        "message_type": "ACCEPT_PROPOSAL",
        "item": {
            "name": product,
            "quantity": quantity,
            "price": price
        },
        "total_cost": total_cost
    }


@mcp.tool
def check_product(product: str, requested_quantity: int) -> dict:
    """
    Checks whether the requested quantity of a product is available.
    """

    if requested_quantity <= 0:
        return {"message": "Quantity must be greater than zero."}

    product = get_product(product)

    if product is None:
        return {"message": f"Product {product} is not available in the warehouse."}

    quantity = float(product["quantity"])
    price = float(product["price"])

    if requested_quantity > quantity:
        return {"message": (f"Warehouse does not have enough of {product['name']}")}

    total_price = requested_quantity * price

    return {
        "available": True,
        "product": product["name"],
        "requested_quantity": requested_quantity,
        "available_quantity": quantity,
        "unit_price": price,
        "total_price": total_price
    }


@mcp.tool
def products_status() -> list[dict]:
    """
    Returns the status of all products in the warehouse.
    """
    connection = get_connection()
    try:
        cursor = connection.cursor(dictionary=True)
        cursor.execute(
            "SELECT name, quantity, price FROM warehouse"
        )
        rows = cursor.fetchall()
        return [
            {"name": item["name"],
            "quantity": item["quantity"],
            "price": item["price"]
            }
            for item in rows]
    finally:
        cursor.close()
        connection.close()


@mcp.tool
def stock_info(product: str) -> dict:
    """
    Returns stock information about a specific product.
    """
    product= get_product(product)

    if product is None:
        return {"message": f"Product {product} is not available in the warehouse."}

    return {
        "product": product["name"],
        "available_quantity": product["quantity"],
        "unit_price": product["price"]
    }


@mcp.tool
def get_proposal(sender_id: str,item: str,quantity: int) -> dict:
    """
    Prepare proposal for a specific item and quantity . Returns a message.
    """
    if quantity <= 0:
        return {"message": "Quantity must be greater than zero."}

    product = get_product(item)

    if product is None:
        return {"message": f"Product {item} is not available in the warehouse."}

    quantity_current = product["quantity"]
    price = product["price"]

    if quantity > quantity_current:
        return {"message": (f"Not enough {product['name']} in warehouse, available now {quantity_current}. ")}
    total_cost = quantity * price
    return {
        "sender_id": "H2",
        "receiver_id": sender_id,
        "message_type": "PROPOSAL",
        "item": {
            "name": product["name"],
            "quantity": quantity,
            "price": price
        },
        "total_cost": total_cost
    }


@mcp.tool
def sale_product(sender_id: str,item: str,quantity: int,payment: float) -> dict:
    """
    Sales a product from the warehouse, finalizing the order. Returns a message to be sent to the sender.
    """
    if quantity <= 0:
        return {"message": "Warehouse cannot sell a product with quantity less than or equal to zero."}
    product = get_product(item)

    if product is None:
        return {"message": f"Product {item} is not available."}
    
    available_quantity = product["quantity"]
    if quantity > available_quantity:
        return {"message": (f"Not enough {product['name']} in warehouse, available now {available_quantity}. ")}
   
    total_cost = quantity * product["price"]
    if abs(payment - total_cost) > 0.00001:
        return {"message": (f"Payment amount does not match the expected total cost. Warehouse cannot sell the product")}
    connection = get_connection()
    try:
        cursor = connection.cursor()
        cursor.execute(
            """
            UPDATE warehouse
            SET quantity = quantity - %s
            WHERE name = %s AND quantity >= %s
            """,
            (quantity, item, quantity)
        )
        if cursor.rowcount == 0:
            connection.rollback()
            return { "message": "Warehouse cannot sell the product." }
        connection.commit()
    finally:
        cursor.close()
        connection.close()
    return {
        "sender_id": "H2",
        "receiver_id": sender_id,
        "message_type": "DELIVERY",
        "item": {
            "name": product["name"],
            "quantity": quantity,
            "price": product["price"]
        },
        "total_cost": payment
    }
@mcp.tool
def receive_product_from_producer(product:str,bought_quantity:int,price:float) ->  dict:
    '''
    Updates the warehouse stock based on the received products from the producer.
    '''
    product = get_product(product)
    if bought_quantity <= 0:
        return {"message": "Bought quantity must be greater than zero."}
    if product is None:
        return {"message": f"Product {product} is not available in the warehouse."}
    current_quantity = product["quantity"]
    connection = get_connection()
    total_cost = bought_quantity * price
    try:
        cursor = connection.cursor(dictionary=True)
        cursor.execute(
            """
            UPDATE warehouse
            SET quantity = quantity + %s
            WHERE name = %s
            """,
            (bought_quantity, product["name"]))

        
        connection.commit()
        return {
                "sender_id": "P",
                "receiver_id": "H2",
                "message_type": "DELIVERY",
                "item": {
                    "name": product["name"],
                    "quantity": bought_quantity,
                    "price": price
                },
                "total_cost": total_cost
            }

    finally:
        cursor.close()
        connection.close()

In [20]:
tools = [
    request_offer,
    accept_p_offer,
    check_product,
    products_status,
    stock_info,
    get_proposal,
    sale_product,
    receive_product_from_producer
]
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    system_prompt=SYSTEM_PROMPT,
    tools=tools,
    checkpointer=InMemorySaver(),
)
config: RunnableConfig={"configurable":{"thread_id":"1"}}